In [ ]:
'''
    Code to compute the relative and absolute genetic risk across contexts for a given phenotype
'''

import pandas as pd
import numpy as np
import random
import time

from options.options import Options
import util.util as util
import util.pre_process as pre
import util.bootstrap_tools as btool

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as plticker
import numpy as np

import torch
from scipy.stats import norm

MIN_VAL = 20 # minimum number of cases in strata for analysis, otherwise skip

phen_label = 'MDD' # set the phenotype to be examined

# Function to calculate relative and absolute genetic risk within strata
def gen_results(pgs, phen, pgs_res, phen_res, top_or_null=None, quant_val=None):
    results = {}

    # Calculate observed R² between residualized PGS and phenotype
    results_ = btool.efficient_r2(pgs_res, phen_res)
    results['high_pgs'] = (pgs > pgs_emerge_cutoff).mean()  # Proportion above PGS threshold
    results['r2_obs'] = results_['r2']
    results['prev'] = phen.mean(1)  # Prevalence per phenotype vector

    # Convert observed R² to liability scale
    results['r2_liab'] = btool.effecient_r2_liab(results['r2_obs'], results['prev'])

    # Convert R² to odds ratio and absolute risk based on top quantile
    results_ = btool.effecient_r2_to_risk(results['r2_obs'], results['prev'], emerge_cutoff, quant_val)
    results['top_or'] = results_['odds_ratio']
    results['top_ar'] = results_['abs_risk']
    results['top_ar_control'] = results_['abs_risk_control']
    results['quant_val'] = np.asarray(results_['quant_val'])

    # Compute overall odds ratio
    results['or'] = btool.efficient_odds_ratio(results['r2_obs'], results['prev'])

    # If null top OR is provided, compute absolute risk under null
    if top_or_null is not None:
        odds = results['top_ar_control'] / (1 - results['top_ar_control'])
        results['top_ar_null'] = top_or_null * odds / (1 + top_or_null * odds)

    return results

# Function to compute bootstrapped distributions of risk metrics
def gen_random_results(pgs_, phen_, x, y, n_bootstrap, max_rand, var_names, top_or_null, quant_val=None):
    results = {}
    for var_name in var_names:
        results[var_name] = np.full((n_bootstrap,), np.nan)  # Initialize result arrays

    n_rand_groups = (n_bootstrap // max_rand)  # Number of bootstrap batches
    for r_group in range(n_rand_groups + 1):
        r_beg = r_group * opt.max_random
        r_end = min((r_group + 1) * max_rand, n_bootstrap)
        n_rand = r_end - r_beg

        if r_beg == r_end:
            continue

        # Sample bootstrap indices
        rand_idx = torch.randint(0, len(pgs_), size=(n_rand, len(pgs_)), dtype=torch.int32) # use torch as it is a faster way to generate random ints. Can be changed.

        # Subsample data using bootstrap indices
        pgs_rand = pgs_[rand_idx]
        phen_rand = phen_[rand_idx]
        x_rand = x[rand_idx]
        y_rand = y[rand_idx]

        # Compute risk metrics for each bootstrap sample
        results_ = gen_results(pgs_rand, phen_rand, x_rand, y_rand, top_or_null, quant_val)
        for key in results:
            results[key][r_beg:r_end] = results_[key]

    return results

opt = Options()
opt.initialize()

# Determine eMERGE cutoff points
emerge_cutoff = opt.phen_params[phen_label]['emerge_cut']
pgs_emerge_cutoff = norm.ppf(1-emerge_cutoff)

# load PGS and covar files
pgs_df = pd.read_csv(opt.prs_file, delim_whitespace=True, index_col='IID')
covars_df = pd.read_csv(opt.env_file, sep='\t', index_col='IID')

pc_cols = ['chip']+[df_col for df_col in covars_df.columns if df_col.startswith('PC')]
pcs_df = covars_df[pc_cols].iloc[:, :11]
covars_df = covars_df[opt.covars]

phen = pgs_df[opt.phen_params[phen_label]['phen']]
phen = phen.dropna()
pgs = pgs_df[opt.phen_params[phen_label]['pgs']].loc[phen.index]
pgs = (pgs-pgs.mean())/pgs.std()
if opt.phen_params[phen_label]['pgs_flip']:
    pgs = -1*pgs
covars_df = covars_df.loc[phen.index]

# impute values in matrix
covars_imputed_df = pre.impute_covars(covars_df)
covars_imputed_df, _, _ = pre.standardize_covars(covars_imputed_df)
covars = pd.concat([covars_imputed_df, pcs_df], axis=1)
covars = covars.loc[pgs.index]

# regress out covars from phenotype and pgs over the full sample
phen_res, _ = pre.lin_regress_out(covars, phen)
pgs_res, _ = pre.lin_regress_out(covars, pgs)

# Get indices of subpopulations
bin_defs = opt.bin_defs
bounds, labels, label_map = util.create_bins_and_indices(covars_df, bin_defs)

util.create_folder(f'results/{phen_label}')

print('\n')
print(f'Total: {len(phen)}')
print(f'Cases: {phen.sum()}')
for sex_label in ['Female', 'Male']:
    n_sex = len(phen.loc[label_map['Sex: '+sex_label]])
    n_cases = phen.loc[label_map['Sex: '+sex_label]].sum()
    print(f'{sex_label} Cases: {n_cases}/{n_sex}')

for k in label_map.keys():
    print(len(label_map[k]), ': ', k,)

del pgs_df, covars_imputed_df, covars # delete to free memory

# Compute relative and genetic risk over entire sample

In [ ]:
'''
Variables stored -
    high-pgs: portion of individuals above eMERGE PGS threshold within context (range: 0-1)
    prev: prevalence of disease within context (range: 0-1)
    r2_obs: Proportion of variance in disease status explained by PGS on the observed scale within a context
    r2_liab: Proportion of variance explained in disease status by PGS on the liability scale (accounting for disease prevalence) within a context
    or: odds ratio for disease per 1 std increase in PGS within a context
    top_or: odds ratio of disease for high-PGS individuals within a context
    top_ar: absolute risk of disease for high-PGS individuals within a context
    top_ar_control: absolute risk of disease for low-PGS (i.e. not high-PGS) individuals within a context
    top_ar_null: absolute risk of disease for high-PGS individuasl using top_or from overall population
'''

vars = ['high_pgs', 'prev', 'r2_obs', 'r2_liab', 
        'or', 'top_or', 'top_ar', 'top_ar_control', 'top_ar_null']

all_results, bot_results, top_results = {}, {}, {}
pgs_, phen_, x, y = pgs.values, phen.values, pgs_res.values, phen_res.values

all_results['n'] = len(pgs_)
results_ = gen_results(pgs_[None, :], phen_[None, :], x[None, :], y[None, :], None)
all_results.update({key: results_[key].item() for key in vars if key in results_})
TOP_OR_NULL = all_results.get('top_or')
np.save(f'results/{phen_label}/all_results.npy', all_results)

# Compute relative and genetic risk over one-way and two-way intersectional contexts

In [ ]:
# Initialize arrays
n = len(labels)

# Using dictionary instead of globals() to store arrays
results = {var: np.full((n, n), np.nan) for var in vars}
bootstrap_results = {}
n_idx = np.full((n, n), np.nan)

## Intersect analysis
for i in range(n):
    for j in range(i+1):
        idx_org = np.intersect1d(label_map[labels[i]], label_map[labels[j]])
        
        n_idx[i, j] = n_idx[j, i] = len(idx_org)

        pgs_ = pgs.loc[idx_org].values
        phen_ = phen.loc[idx_org].values
        x = pgs_res.loc[idx_org].values
        y = phen_res.loc[idx_org].values

        if phen_.sum() <= MIN_VAL: # skip analysis if case count less than MIN_VAL (default 20 cases)
            continue

        results_ = gen_results(pgs_[None,:], phen_[None,:], x[None,:], y[None,:], TOP_OR_NULL)
        for key in results_:
            if key not in vars:
                continue
            results[key][i, j] = results[key][j, i] = results_[key].item()

# Save results
for var in vars:
    np.save(f'results/{phen_label}/{var}.npy', results[f'{var}'])
np.save(f'results/{phen_label}/n_idx.npy', n_idx)

# Compute relative and genetic risk over three-way intersectional contexts

In [ ]:
n = len(labels)

n_3way = {}
results_3way = {}

for i in range(n):
    for j in range(i):
        for k in range(j):
            idx_org = np.intersect1d(np.intersect1d(label_map[labels[i]], label_map[labels[j]]), label_map[labels[k]])

            pgs_ = pgs.loc[idx_org].values
            phen_ = phen.loc[idx_org].values
            x = pgs_res.loc[idx_org].values
            y = phen_res.loc[idx_org].values

            if phen_.sum() <= MIN_VAL: # skip analysis if case count less than MIN_VAL (default 20 cases)
                continue

            n_3way[(i,j,k)] = len(pgs_)
            results_3way[(i,j,k)] = {}

            results_ = gen_results(pgs_[None,:], phen_[None,:], x[None,:], y[None,:], TOP_OR_NULL)
            for key in results_:
                results_3way[(i,j,k)][key] = results_[key].item()
            results_3way[(i,j,k)]['n'] = len(y)

np.save(f'results/{phen_label}/results_3way.npy', results_3way)
np.save(f'results/{phen_label}/n_3way.npy', n_3way)

# Compute bootstrapped distributions of the minimum and maximum odds ratios (ORs) within each variable and variable intersection

In [ ]:
import time
start_time = time.time()

for i in range(len(bounds)-1):
    for j in range(i+1):
        bootstrap_results[(i,j)] = {'min':{}, 'max':{}}
        i_strt = bounds[i]
        i_end = bounds[i+1]
        j_strt = bounds[j]
        j_end = bounds[j+1]

        context = results['top_or'][i_strt:i_end, j_strt:j_end]

        # find indeices corresponding to min and max OR
        min_idx = np.unravel_index(np.nanargmin(context), context.shape)
        max_idx = np.unravel_index(np.nanargmax(context), context.shape)

        min_idx = (min_idx[0]+i_strt, min_idx[1]+j_strt)
        max_idx = (max_idx[0]+i_strt, max_idx[1]+j_strt)

        for extreme, idx in zip(['min', 'max'], [min_idx, max_idx]):
            i_idx = idx[0]
            j_idx = idx[1]
            
            idx_org = np.intersect1d(label_map[labels[i_idx]], label_map[labels[j_idx]])
            if len(idx_org) == 0:
                continue

            pgs_ = pgs.loc[idx_org].values
            phen_ = phen.loc[idx_org].values
            x = pgs_res.loc[idx_org].values
            y = phen_res.loc[idx_org].values

            results_rand_ = gen_random_results(pgs_, phen_, x, y, opt.n_bootstrap, opt.max_random, vars, TOP_OR_NULL, None)
            
            bootstrap_results[(i,j)][extreme] = {'index': (i_idx, j_idx), 'bootstrap': {}}
            for key in results_rand_:
                bootstrap_results[(i,j)][extreme]['bootstrap'][key] = results_rand_[key]
    print(f"{i}: - time: {time.time() - start_time} seconds")
 
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time} seconds")

np.save(f'results/{phen_label}/bootstrap.npy', bootstrap_results)

# Compute bootstrapped distributions of the minimum and maximum odds ratios (ORs) across three-way contexts

In [ ]:
min_idx_3way = min(results_3way, key=lambda k: results_3way[k]['top_or'])
max_idx_3way = max(results_3way, key=lambda k: results_3way[k]['top_or'])

min_max_3way = {}
for key, idx_ls in zip(['min', 'max'], [min_idx_3way, max_idx_3way]):
    print(key)
    i, j, k = idx_ls

    idx_org = np.intersect1d(np.intersect1d(label_map[labels[i]], label_map[labels[j]]), label_map[labels[k]])

    pgs_ = pgs.loc[idx_org].values
    phen_ = phen.loc[idx_org].values
    x = pgs_res.loc[idx_org].values
    y = phen_res.loc[idx_org].values

    results_rand_ = gen_random_results(pgs_, phen_, x, y, opt.n_bootstrap, opt.max_random, vars[:-1], None)
    
    min_max_3way[key] = results_rand_
    min_max_3way[key]['index'] = idx_ls
np.save(f'results/{phen_label}/minmax_3way.npy', min_max_3way)

In [ ]:
results = {}
results['top_or'] = np.load(f'results/{phen_label}/top_or.npy')

# Calculate the maximum OR %Δ across variables and variable intersections

In [ ]:
def get_pdiff(var_min, var_max):
    var_range = (var_max-var_min)
    var_pdiff = 100*((var_max/var_min)-1)

    return var_range, var_pdiff

var_range = np.full((6,6,opt.n_bootstrap+1), np.nan)
var_pdiff = np.full((6,6,opt.n_bootstrap+1), np.nan)
for i in range(len(bounds)-1):
    for j in range(i+1):
        max_idx = bootstrap_results[(i,j)]['max']['index']
        min_idx = bootstrap_results[(i,j)]['min']['index']

        max_odds = results['top_or'][max_idx]
        min_odds = results['top_or'][min_idx]
        
        max_bootstrap = bootstrap_results[(i,j)]['max']['bootstrap']['top_or']
        min_bootstrap = bootstrap_results[(i,j)]['min']['bootstrap']['top_or']

        var_range_, var_pdiff_ = get_pdiff(min_odds, max_odds)
        var_range_bootstrap, var_pdiff_bootstrap = get_pdiff(min_bootstrap, max_bootstrap)

        var_range[i,j,0] = var_range[j,i,0] =  var_range_
        var_pdiff[i,j,0] = var_pdiff[j,i,0] = var_pdiff_
        
        var_range[i,j,1:] = var_range[j,i,1:] =  var_range_bootstrap
        var_pdiff[i,j,1:] = var_pdiff[j,i,1:] = var_pdiff_bootstrap

np.save(f'results/{phen_label}/or_range_2way.npy', var_range)
np.save(f'results/{phen_label}/or_pdiff_2way.npy', var_pdiff)

# Calculate significance of maximum OR %Δ across variables and variable intersections

In [ ]:
n_vars = var_pdiff.shape[0]
n_contexts = (n_vars)*(n_vars+1)//2

zero_pval = np.full((n_vars, n_vars), np.nan)
parent_pval = zero_pval.copy()
max_pval = zero_pval.copy()

for i in range(n_vars):
    samps = var_pdiff[i, i]
    _, p_zero = util.calc_empirical_pval(samps[1:], two_tail=False)
    zero_pval[i, i] = p_zero
    for j in range(i + 1, n_vars):
        samps = var_pdiff[i, j]
        samps_p1 = var_pdiff[i, i]
        samps_p2 = var_pdiff[j, j]
        
        if np.isnan(samps.mean()):
            continue
            
        _, p_zero = util.calc_empirical_pval(samps[1:], two_tail=False)
        _, p1_zero = util.calc_empirical_pval(samps_p1[1:], two_tail=False)
        _, p2_zero = util.calc_empirical_pval(samps_p2[1:], two_tail=False)
        zero_pval[i, j] = zero_pval[j, i] = p_zero

        _, p_parent1 = util.calc_empirical_pval(samps[1:]-samps_p1[0], two_tail=False)
        _, p_parent2 = util.calc_empirical_pval(samps[1:]-samps_p2[0], two_tail=False)

        if p_zero<=.05/n_contexts:
            parent_pval[i, j] = parent_pval[j, i] = np.maximum(p_parent1, p_parent2)


np.save(f'results/{phen_label}/pval_zero.npy', zero_pval)
np.save(f'results/{phen_label}/pval_parent.npy', parent_pval)

# Find the top 5 and bottom 5 largest differences in context aware vs. context unaware PGS absolute risk estimates

In [ ]:
top_ar = np.load(f'results/{phen_label}/top_ar.npy')
top_ar_null = np.load(f'results/{phen_label}/top_ar_null.npy')

diff_matrix = top_ar - top_ar_null
rows, cols = np.tril_indices(diff_matrix.shape[0], k=-1)
lower_triangle_values = diff_matrix[rows, cols]

valid_mask = ~np.isnan(lower_triangle_values)
valid_rows, valid_cols = rows[valid_mask], cols[valid_mask]
valid_values = lower_triangle_values[valid_mask]
top_indices = np.argsort(valid_values)[-5:]
bot_indices = np.argsort(valid_values)[:5]

top_pairs = list(zip(valid_rows[top_indices], valid_cols[top_indices]))
bot_pairs = list(zip(valid_rows[bot_indices], valid_cols[bot_indices]))

print('Top AR Diff:', diff_matrix[top_pairs[-1]])
print('Bot AR Diff:',diff_matrix[bot_pairs[0]])

# Bootstrap relative and absolute genetic risks from the contexts found in the cell above

In [ ]:
# Initialize arrays
n = len(labels)

# Using dictionary instead of globals() to store arrays
top_strata_results = {}
bot_strata_results = {}

## Intersect analysis
for idx_n, (i, j) in enumerate(top_pairs):
    idx_i = len(top_pairs) - idx_n
    print('Doing top:', idx_n)
    top_strata_results[idx_i] = {'idx': (i,j)}
    idx_org = np.intersect1d(label_map[labels[i]], label_map[labels[j]])
    
    n_idx[i, j] = n_idx[j, i] = len(idx_org)

    pgs_ = pgs.loc[idx_org].values
    phen_ = phen.loc[idx_org].values
    x = pgs_res.loc[idx_org].values
    y = phen_res.loc[idx_org].values

    if phen_.sum() <= MIN_VAL:
        continue

    results_ = gen_results(pgs_[None,:], phen_[None,:], x[None,:], y[None,:], TOP_OR_NULL)
    for key in results_:
        if key not in vars:
            continue
        top_strata_results[idx_i][key] = results_[key].item()
    
    results_rand_ = gen_random_results(pgs_, phen_, x, y, opt.n_bootstrap, opt.max_random, vars, TOP_OR_NULL, None)
    for key in results_rand_:
        top_strata_results[idx_i][key+'_bootstrap'] = results_rand_[key]
    
    pval = util.calc_empirical_pval(results_rand_['top_ar']-results_['top_ar_null'], two_tail=True)
    top_strata_results[idx_i]['pval'] = pval

    

## Intersect analysis
for idx_n, (i, j) in enumerate(bot_pairs):
    idx_i = len(bot_pairs) - idx_n
    print('Doing bot:', idx_n)
    bot_strata_results[idx_i] = {'idx': (i,j)}
    idx_org = np.intersect1d(label_map[labels[i]], label_map[labels[j]])
    
    n_idx[i, j] = n_idx[j, i] = len(idx_org)

    pgs_ = pgs.loc[idx_org].values
    phen_ = phen.loc[idx_org].values
    x = pgs_res.loc[idx_org].values
    y = phen_res.loc[idx_org].values

    if phen_.sum() <= MIN_VAL:
        continue

    results_ = gen_results(pgs_[None,:], phen_[None,:], x[None,:], y[None,:], TOP_OR_NULL)
    for key in results_:
        if key not in vars:
            continue
        bot_strata_results[idx_i][key] = results_[key].item()
    
    results_rand_ = gen_random_results(pgs_, phen_, x, y, opt.n_bootstrap, opt.max_random, vars, TOP_OR_NULL, None)
    for key in results_rand_:
        bot_strata_results[idx_i][key+'_bootstrap'] = results_rand_[key]

    pval = util.calc_empirical_pval(results_rand_['top_ar']-results_['top_ar_null'], two_tail=True)
    bot_strata_results[idx_i]['pval'] = pval

np.save(f'results/{phen_label}/top_ar_diff.npy', top_strata_results)
np.save(f'results/{phen_label}/bot_ar_diff.npy', bot_strata_results)